### Import libraries

In [2]:
%matplotlib inline
import os
import sys
import csv
import torch
import pathlib
import matplotlib
import numpy as np
import pandas as pd
import seaborn as sb

### Import Autoencoder class

In [4]:
folder_path = str(pathlib.Path().resolve().parents[1])
print("Folder path =", folder_path)
sys.path.append(folder_path + '/Notebooks/AE_class/')
from Conv_AE_weebots_downsampled import *

Folder path = C:\Users\specs\Desktop\Robotology\Repos\HPC-VTA_Autoencoder


### Check libraries versions

In [6]:
print(f"torch version: {torch.__version__}")
print(f"numpy version: {np.__version__}")
print(f"matplotlib version: {matplotlib.__version__}")
print(f"seaborn version: {sb.__version__}")

torch version: 2.2.1+cu121
numpy version: 1.26.4
matplotlib version: 3.8.0
seaborn version: 0.12.2


### Check GPU

In [8]:
print("CUDA is available =", torch.cuda.is_available())
print("CUDA device count =", torch.cuda.device_count())
if torch.cuda.device_count() > 1:
    os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
#print(torch.cuda.current_device())
#print(torch.cuda.device(0))
print("CUDA device name =", torch.cuda.get_device_name(0))

CUDA is available = True
CUDA device count = 2
CUDA device name = NVIDIA RTX A5000


### Hyperparameters

In [10]:
#Model
n_hidden = 200
batch_size = 256
num_epochs = 100
learning_rate = 1e-4
C_factor = 1e3
alpha = 1e5

#Others
experiment_n = 2
trial_t = 0

envs = ['Tmaze_old1', 'Tmaze_old2', 'Tmaze_old3']

In [11]:
def format_dataset(env):
    data_path = folder_path + '/Datasets/' + env
    csv_filename = data_path + "/data.csv"

    img_filename_list = os.listdir(data_path)
    num_images = len(img_filename_list) -1 

    return data_path, csv_filename, num_images

#####################################################################

def Load_env_dataset(env_idx):
    # Load dataset every T-maze version
    env = envs[env_idx] #Choose environment index
    data_path, csv_filename, num_images = format_dataset(env)

    print("----------  ----------  NEW ENVIRONMENT ----------  ----------")
    print("Dataset Environment    =", env)
    print("Num. Images in Dataset =", num_images)
    print("Dataset Path =", data_path)
    print("CSV path     =", csv_filename)
    
    dataset = load_dataset(data_path)  # this array should have shape (n_samples, 3, 120, 160)

    return dataset, csv_filename

#####################################################################

def update_model_trial(experiment_n, env_idx, trial):
    # Update model name for trial t
    env = envs[env_idx]
    trial_t = trial
    model_filename =  env + '-Trial' + str(trial_t) + 'of' + str(num_epochs) + '-NHidden' + str(n_hidden) + '-BSize' + str(batch_size) + '-C' + str(C_factor) + '-A' + str(alpha) + '-LR' + str(learning_rate) + '.pth'
    foldername = 'Control-Exp' + str(experiment_n) + '-NEpisodes' + str(num_epochs) + '-NHidden' + str(n_hidden) + '-BSize' + str(batch_size) + '-C' + str(C_factor) + '-A' + str(alpha) + '-LR' + str(learning_rate)

    model_folder_path = folder_path + '/Models/' + foldername
    model_path = model_folder_path + '/' + model_filename    

    model = Conv_AE(n_hidden=n_hidden)
    model.load_state_dict(torch.load(model_path))

    return model

#####################################################################

def update_data(csv_filename):
    data = pd.read_csv(csv_filename)
    #data = data[:num_images]
    
    position = [] 
    pos_step = []
    
    for i in range(len(data)):
        pos_step = [data.X[i], data.Y[i]]
        position.append(pos_step)
        
    position = np.array(position)
    print("Dataset Shape  =", dataset.shape)
    print("Position Shape =", position.shape)

    return data, position

In [12]:
def init_ratemap_cache(path, n_checkpoints, n_units=200, h=50, w=50, dtype=np.float32):
    """
    Create a new memmap .npy file initialized with NaNs.
    """
    arr = np.lib.format.open_memmap(
        path, mode="w+", dtype=dtype, shape=(n_checkpoints, n_units, h, w)
    )
    arr[:] = np.nan
    del arr  # flush

def open_ratemap_cache(path, n_checkpoints, n_units=200, h=50, w=50, dtype=np.float32):
    """
    Open an existing cache (r+) or create it if missing.
    Returns a memmap array you can slice-assign into.
    """
    if not os.path.exists(path):
        init_ratemap_cache(path, n_checkpoints, n_units, h, w, dtype)
    arr = np.lib.format.open_memmap(path, mode="r+")
    
    # Optional: sanity check shape
    expected = (n_checkpoints, n_units, h, w)
    if tuple(arr.shape) != expected:
        raise ValueError(f"Cache shape mismatch at {path}. Found {arr.shape}, expected {expected}")
    return arr

In [13]:
CACHE_DIR = folder_path + "/Models/Control-Exp" + str(experiment_n) + "_cache_ratemaps"
os.makedirs(CACHE_DIR, exist_ok=True)

N_BINS = 50
FILTER_WIDTH = 3


# Trials/checkpoints are 0..num_epochs inclusive in your logic (since you use num_epochs+1 sometimes)
n_checkpoints = num_epochs * len(envs) + 1

cache_path = os.path.join(CACHE_DIR, "ratemaps.npy")
cache_arr = open_ratemap_cache(
    cache_path,
    n_checkpoints=n_checkpoints,
    n_units=n_hidden,
    h=N_BINS,
    w=N_BINS,
    dtype=np.float32
)

global_t = 0

for m in range(len(envs)):
    dataset, csv_filename = Load_env_dataset(m)
    _, position = update_data(csv_filename)

    env_trial_range = num_epochs +1 if m == 0 else num_epochs

    for t in range(env_trial_range):
        env_t = t+1 if m != 0 else t
        print(f"Tmaze={m}, Trial={env_t}, Global={global_t}")

        model = update_model_trial(experiment_n, m, env_t)
        embeddings = get_latent_vectors(dataset, model, batch_size=batch_size)

        rmaps = ratemaps(embeddings, position, n_bins=N_BINS, filter_width=FILTER_WIDTH)
        rmaps = np.asarray(rmaps, dtype=np.float32)

        # skip if already written
        if not np.isnan(cache_arr[global_t, 0, 0, 0]):
            continue

        cache_arr[global_t] = rmaps

        global_t = global_t + 1

del cache_arr  # flush once at the end

----------  ----------  NEW ENVIRONMENT ----------  ----------
Dataset Environment    = Tmaze_old1
Num. Images in Dataset = 39998
Dataset Path = C:\Users\specs\Desktop\Robotology\Repos\HPC-VTA_Autoencoder/Datasets/Tmaze_old1
CSV path     = C:\Users\specs\Desktop\Robotology\Repos\HPC-VTA_Autoencoder/Datasets/Tmaze_old1/data.csv
Dataset Shape  = (39998, 120, 160, 3)
Position Shape = (39998, 2)
Tmaze=0, Trial=0, Global=0
Tmaze=0, Trial=1, Global=1
Tmaze=0, Trial=2, Global=2
Tmaze=0, Trial=3, Global=3
Tmaze=0, Trial=4, Global=4
Tmaze=0, Trial=5, Global=5
Tmaze=0, Trial=6, Global=6
Tmaze=0, Trial=7, Global=7
Tmaze=0, Trial=8, Global=8
Tmaze=0, Trial=9, Global=9
Tmaze=0, Trial=10, Global=10
Tmaze=0, Trial=11, Global=11
Tmaze=0, Trial=12, Global=12
Tmaze=0, Trial=13, Global=13
Tmaze=0, Trial=14, Global=14
Tmaze=0, Trial=15, Global=15
Tmaze=0, Trial=16, Global=16
Tmaze=0, Trial=17, Global=17
Tmaze=0, Trial=18, Global=18
Tmaze=0, Trial=19, Global=19
Tmaze=0, Trial=20, Global=20
Tmaze=0, Trial=2

In [25]:
batch_training = True

In [ ]:
if batch_training == True:
    total_experiments = 10
    starting_experiment = 3
    current_exp = starting_experiment
    
    while current_exp <= total_experiments:
        CACHE_DIR = folder_path + "/Models/Control-Exp" + str(current_exp) + "_cache_ratemaps"
        os.makedirs(CACHE_DIR, exist_ok=True)
        
        N_BINS = 50
        FILTER_WIDTH = 3
        
        
        # Trials/checkpoints are 0..num_epochs inclusive in your logic (since you use num_epochs+1 sometimes)
        n_checkpoints = num_epochs * len(envs) + 1
        
        cache_path = os.path.join(CACHE_DIR, "ratemaps.npy")
        cache_arr = open_ratemap_cache(
            cache_path,
            n_checkpoints=n_checkpoints,
            n_units=n_hidden,
            h=N_BINS,
            w=N_BINS,
            dtype=np.float32
        )
        
        global_t = 0
        
        for m in range(len(envs)):
            dataset, csv_filename = Load_env_dataset(m)
            _, position = update_data(csv_filename)
        
            env_trial_range = num_epochs +1 if m == 0 else num_epochs
        
            for t in range(env_trial_range):
                env_t = t+1 if m != 0 else t
                print(f"Tmaze={m}, Trial={env_t}, Global={global_t}")
        
                model = update_model_trial(current_exp, m, env_t)
                embeddings = get_latent_vectors(dataset, model, batch_size=batch_size)
        
                rmaps = ratemaps(embeddings, position, n_bins=N_BINS, filter_width=FILTER_WIDTH)
                rmaps = np.asarray(rmaps, dtype=np.float32)
        
                # skip if already written
                if not np.isnan(cache_arr[global_t, 0, 0, 0]):
                    continue
        
                cache_arr[global_t] = rmaps
        
                global_t = global_t + 1
        
        del cache_arr  # flush once at the end
        
        current_exp += 1

----------  ----------  NEW ENVIRONMENT ----------  ----------
Dataset Environment    = Tmaze_old1
Num. Images in Dataset = 39998
Dataset Path = C:\Users\specs\Desktop\Robotology\Repos\HPC-VTA_Autoencoder/Datasets/Tmaze_old1
CSV path     = C:\Users\specs\Desktop\Robotology\Repos\HPC-VTA_Autoencoder/Datasets/Tmaze_old1/data.csv
Dataset Shape  = (39998, 120, 160, 3)
Position Shape = (39998, 2)
Tmaze=0, Trial=0, Global=0
Tmaze=0, Trial=1, Global=1
Tmaze=0, Trial=2, Global=2
Tmaze=0, Trial=3, Global=3
Tmaze=0, Trial=4, Global=4
Tmaze=0, Trial=5, Global=5
Tmaze=0, Trial=6, Global=6
Tmaze=0, Trial=7, Global=7
Tmaze=0, Trial=8, Global=8
Tmaze=0, Trial=9, Global=9
Tmaze=0, Trial=10, Global=10
Tmaze=0, Trial=11, Global=11
Tmaze=0, Trial=12, Global=12
Tmaze=0, Trial=13, Global=13
Tmaze=0, Trial=14, Global=14
Tmaze=0, Trial=15, Global=15
Tmaze=0, Trial=16, Global=16
Tmaze=0, Trial=17, Global=17
Tmaze=0, Trial=18, Global=18
Tmaze=0, Trial=19, Global=19
Tmaze=0, Trial=20, Global=20
Tmaze=0, Trial=2